<a href="https://colab.research.google.com/github/hsandmann/biblio/blob/main/material/handouts/gradiente-descendente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradiente descendente, na prática

**Machine Learning · Insper**

Na aula passada a regressão foi resolvida de uma vez, pelas equações normais. Aqui fazemos
o contrário: **chegamos à mesma resposta andando**, um passo de cada vez.

O roteiro:

1. descer uma ladeira de uma variável só, para ver a regra funcionando;
2. mexer na taxa de aprendizado até o método quebrar;
3. treinar uma reta com três pontos e conferir os números da tabela da aula;
4. olhar a paisagem da perda e ver por que centrar os dados acelera tudo;
5. escrever o gradiente em uma linha e usá-lo em 20 mil casas da Califórnia;
6. trocar o conjunto inteiro por lotes pequenos (mini-batch).

Só `numpy` e `matplotlib`, mais `pandas` para ler um CSV. Rode as células **em ordem**.
Onde aparecer **✋ Aposte antes de rodar**, pare e responda antes de executar a célula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
LARANJA, AZUL, VERMELHO = '#f0883e', '#1F6B8E', '#9E3B2E'
print("pronto")

## 1. Descer uma ladeira

Comece pela perda mais simples possível, com um parâmetro só:

$$J(w) = (w - 3)^2$$

Você sabe que o mínimo é $w = 3$. O algoritmo não sabe. Tudo o que ele consegue fazer é
**medir a inclinação** onde está,

$$\frac{dJ}{dw} = 2(w - 3),$$

e dar um passo contra ela:

$$w \leftarrow w - \eta \, \frac{dJ}{dw}$$

O $\eta$ (eta) é a **taxa de aprendizado**: o tamanho do passo.

In [ ]:
def J(w):
    return (w - 3)**2

def dJ(w):
    return 2*(w - 3)

w = 0.0      # palpite inicial
eta = 0.1    # taxa de aprendizado

print(f"{'passo':>5} {'w':>8} {'J(w)':>9} {'dJ/dw':>8}")
for passo in range(11):
    print(f"{passo:>5} {w:>8.4f} {J(w):>9.4f} {dJ(w):>8.4f}")
    w = w - eta * dJ(w)

Repare na última coluna. O $\eta$ ficou fixo em 0,1, mas os passos **encolhem sozinhos**:
perto do fundo a ladeira é mais plana, a derivada é menor e o passo, que é proporcional a
ela, também. Ninguém precisou avisar o algoritmo de que ele estava chegando.

In [ ]:
def descer(w_inicial, eta, passos):
    ws = [w_inicial]
    for _ in range(passos):
        ws.append(ws[-1] - eta * dJ(ws[-1]))
    return np.array(ws)

caminho = descer(0.0, eta=0.1, passos=10)

ws = np.linspace(-1, 7, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(ws, J(ws), color=AZUL, lw=2, label='J(w) = (w − 3)²')
plt.plot(caminho, J(caminho), 'o-', color=LARANJA, ms=6, lw=1, label='passos')
plt.annotate('início', (caminho[0], J(caminho[0])), textcoords='offset points', xytext=(8, -4))
plt.xlabel('w'); plt.ylabel('J(w)'); plt.grid(alpha=.3); plt.legend()
plt.title('η = 0,1: passos cada vez menores'); plt.show()

## 2. A taxa de aprendizado

A próxima célula repete a descida com quatro valores de $\eta$: **0,1 · 0,5 · 0,9 · 1,05**.

> ✋ **Aposte antes de rodar.** Para cada um: converge devagar, converge rápido, fica
> pulando de um lado para o outro, ou explode?

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(10, 6.5))
ws = np.linspace(-8, 14, 300)

for ax, eta in zip(eixos.flat, [0.1, 0.5, 0.9, 1.05]):
    caminho = descer(0.0, eta, passos=12)
    ax.plot(ws, J(ws), color=AZUL, lw=2)
    ax.plot(caminho, J(caminho), 'o-', color=LARANJA, ms=5, lw=1)
    ax.plot(caminho[-1], J(caminho[-1]), 'o', color=VERMELHO, ms=8)
    ax.set_ylim(-5, 130); ax.grid(alpha=.3)
    ax.set_title(f'η = {eta}   →   w após 12 passos = {caminho[-1]:.3f}')

plt.tight_layout(); plt.show()

Quatro comportamentos com a mesma perda e o mesmo algoritmo:

| $\eta$ | o que aconteceu |
|--:|---|
| 0,1 | chega, mas devagar |
| 0,5 | acerta o mínimo **em um passo** |
| 0,9 | passa do ponto, volta, passa de novo… e fecha |
| 1,05 | passa do ponto por mais do que veio: **diverge** |

A conta explica tudo. Subtraia 3 dos dois lados da regra de atualização:

$$w_{t+1} - 3 = w_t - 3 - \eta \cdot 2(w_t - 3) = (1 - 2\eta)\,(w_t - 3)$$

A distância até o mínimo é **multiplicada por $(1 - 2\eta)$ a cada passo**:

- $\eta = 0{,}1$ → fator $0{,}8$: encolhe 20% por passo;
- $\eta = 0{,}5$ → fator $0$: zera no primeiro passo;
- $\eta = 0{,}9$ → fator $-0{,}8$: troca de lado, mas encolhe;
- $\eta = 1{,}05$ → fator $-1{,}1$: troca de lado **e cresce**.

Converge enquanto $|1 - 2\eta| < 1$, ou seja, $0 < \eta < 1$. Grande demais não é "um pouco
pior": é outro regime, em que o método piora a cada passo.

## 3. Agora uma reta: três pontos

Os mesmos três pontos da aula: $(1, 2)$, $(2, 4)$, $(3, 7)$. O modelo tem dois parâmetros,

$$\hat{y}_i = w_0 + w_1 x_i,$$

e a perda é o erro quadrático **médio**:

$$J(w_0, w_1) = \frac{1}{n}\sum_{i=1}^{n} e_i^2, \qquad e_i = y_i - w_0 - w_1 x_i$$

Antes de descer, guarde a resposta certa pela forma fechada da aula passada. Ela vai servir
de gabarito.

In [ ]:
x = np.array([1., 2., 3.])
y = np.array([2., 4., 7.])

X = np.column_stack([np.ones_like(x), x])
w_exato = np.linalg.solve(X.T @ X, X.T @ y)

print(f"gabarito (forma fechada):  w0 = {w_exato[0]:.4f}   w1 = {w_exato[1]:.4f}")

### 3.1 O gradiente

Pela regra da cadeia, com $\partial e_i / \partial w_0 = -1$ e $\partial e_i / \partial w_1 = -x_i$:

$$
\frac{\partial J}{\partial w_0} = -\frac{2}{n}\sum_i e_i
\qquad\qquad
\frac{\partial J}{\partial w_1} = -\frac{2}{n}\sum_i e_i\,x_i
$$

Cada fórmula vira uma linha de código.

In [ ]:
def perda(w0, w1, x, y):
    e = y - (w0 + w1 * x)
    return np.mean(e**2)

def gradiente(w0, w1, x, y):
    n = len(x)
    e = y - (w0 + w1 * x)              # resíduos
    dJ_dw0 = -2/n * np.sum(e)
    dJ_dw1 = -2/n * np.sum(e * x)
    return dJ_dw0, dJ_dw1

### 3.2 Um passo, conferido

Partindo de $w_0 = w_1 = 0$ com $\eta = 0{,}1$. Compare cada número com a tabela
**"Um passo, à mão"** da página da aula.

In [ ]:
w0, w1, eta = 0.0, 0.0, 0.1

print(f"{'iter':>4} {'w0':>8} {'w1':>8} {'J':>8} {'dJ/dw0':>9} {'dJ/dw1':>9}")
for it in range(4):
    g0, g1 = gradiente(w0, w1, x, y)
    print(f"{it:>4} {w0:>8.3f} {w1:>8.3f} {perda(w0, w1, x, y):>8.3f} {g0:>9.3f} {g1:>9.3f}")
    w0 = w0 - eta * g0
    w1 = w1 - eta * g1

Três coisas para notar na tabela:

- **o primeiro passo faz quase todo o trabalho**: $J$ cai de 23 para 0,63; depois o progresso se arrasta;
- **os gradientes trocam de sinal** entre as iterações 1 e 2: o passo passou do mínimo, como o $\eta = 0{,}9$ da seção 2;
- **$w_1$ recebe um empurrão maior que $w_0$**, porque cada resíduo entra na conta dele multiplicado por $x_i$. Guarde isso para a seção 4.

### 3.3 O laço inteiro

Empacote o laço numa função que guarda o histórico e deixe rodar 2000 iterações.

In [ ]:
def treinar(x, y, eta, iteracoes, w0=0.0, w1=0.0):
    historico = [(w0, w1, perda(w0, w1, x, y))]
    for _ in range(iteracoes):
        g0, g1 = gradiente(w0, w1, x, y)
        w0, w1 = w0 - eta * g0, w1 - eta * g1
        historico.append((w0, w1, perda(w0, w1, x, y)))
    return np.array(historico)          # colunas: w0, w1, J

hist = treinar(x, y, eta=0.1, iteracoes=2000)

print(f"gradiente descendente:  w0 = {hist[-1, 0]:.4f}   w1 = {hist[-1, 1]:.4f}")
print(f"forma fechada:          w0 = {w_exato[0]:.4f}   w1 = {w_exato[1]:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

ax1.plot(hist[:, 2], color=AZUL, lw=2)
ax1.set_yscale('log'); ax1.set_xscale('symlog', linthresh=10); ax1.set_xlim(0, 2000)
ax1.set_xlabel('iteração'); ax1.set_ylabel('J (escala log)')
ax1.set_title('curva de perda'); ax1.grid(alpha=.3)

xs = np.linspace(0.5, 3.5, 50)
for it, alpha in [(0, .15), (1, .3), (2, .45), (5, .6), (20, .8), (2000, 1)]:
    ax2.plot(xs, hist[it, 0] + hist[it, 1] * xs, color=AZUL, alpha=alpha, lw=2,
             label=f'iteração {it}')
ax2.scatter(x, y, s=60, color=LARANJA, zorder=3)
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_title('a reta andando até o ajuste')
ax2.legend(fontsize=8); ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()

A perda **não vai a zero**: ela para em $J \approx 0{,}056$. Três pontos que não estão
alinhados não cabem numa reta, então sobra um erro que nenhum $w$ elimina. Chegar a esse
piso é chegar à forma fechada.

## 4. A paisagem que estamos descendo

Cada par $(w_0, w_1)$ é uma reta candidata, e $J(w_0, w_1)$ é o erro dela. Desenhe isso
como um mapa de curvas de nível e o gradiente descendente vira um caminho nesse mapa.

A próxima célula desenha dois mapas lado a lado:

- **esquerda**: os dados como estão, $x = 1, 2, 3$;
- **direita**: os mesmos dados com $x$ **centrado**, $x - \bar{x} = -1, 0, 1$.

Mesmo $\eta = 0{,}1$, mesmo ponto de partida, 30 passos em cada.

In [ ]:
def mapa(ax, x, y, hist, titulo):
    X = np.column_stack([np.ones_like(x), x])
    w_ot = np.linalg.solve(X.T @ X, X.T @ y)
    W0, W1 = np.meshgrid(np.linspace(w_ot[0] - 4, w_ot[0] + 4, 200),
                         np.linspace(w_ot[1] - 3.5, w_ot[1] + 3.5, 200))
    E = y[:, None, None] - (W0 + W1 * x[:, None, None])
    ax.contour(W0, W1, np.mean(E**2, axis=0), levels=np.geomspace(0.06, 60, 14),
               colors=AZUL, linewidths=.8, alpha=.6)
    ax.plot(hist[:, 0], hist[:, 1], 'o-', color=LARANJA, ms=3, lw=1)
    ax.plot(*w_ot, '*', color=VERMELHO, ms=15, label='ótimo')
    ax.set_xlabel('w0 (intercepto)'); ax.set_ylabel('w1 (inclinação)')
    ax.set_title(titulo); ax.legend(loc='upper right')

x_c = x - x.mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
mapa(ax1, x,   y, treinar(x,   y, 0.1, 30), 'x original: vale estreito')
mapa(ax2, x_c, y, treinar(x_c, y, 0.1, 30), 'x centrado: quase uma tigela redonda')
plt.tight_layout(); plt.show()

Sem centrar, as curvas de nível formam um **vale longo e inclinado**. Com todos os $x$
positivos, mexer na inclinação também sobe e desce a reta inteira, então $w_0$ e $w_1$
ficam acoplados. O gradiente aponta quase atravessado ao vale: o primeiro passo cruza o
vale de uma vez e, dali em diante, o caminho **se arrasta pelo fundo** — em 30 passos ainda
não chegou à estrela.

Centrado, os dois parâmetros se desacoplam: as curvas viram quase círculos e o caminho vai
praticamente reto até o ótimo.

Isso muda também **quanto $\eta$ o problema aguenta**.

> ✋ **Aposte antes de rodar.** Com $\eta = 0{,}5$, qual das duas versões converge?

In [ ]:
def iteracoes_ate_convergir(x, y, eta, tol=1e-3, maximo=10_000):
    X = np.column_stack([np.ones_like(x), x])
    w_ot = np.linalg.solve(X.T @ X, X.T @ y)
    w0 = w1 = 0.0
    for it in range(maximo):
        if max(abs(w0 - w_ot[0]), abs(w1 - w_ot[1])) < tol:
            return f"{it} iterações"
        if abs(w0) + abs(w1) > 1e6:
            return "DIVERGIU"
        g0, g1 = gradiente(w0, w1, x, y)
        w0, w1 = w0 - eta * g0, w1 - eta * g1
    return f"não chegou em {maximo}"

print(f"{'η':>5}   {'x original':>18}   {'x centrado':>18}")
for eta in [0.05, 0.1, 0.15, 0.2, 0.5]:
    print(f"{eta:>5}   {iteracoes_ate_convergir(x, y, eta):>18}   {iteracoes_ate_convergir(x_c, y, eta):>18}")

Com os dados originais, $\eta$ acima de ~0,18 já diverge e o melhor que se consegue são
centenas de iterações. Centrado, $\eta = 0{,}5$ resolve em **8**.

A forma fechada devolve a mesma reta nos dois casos e não liga para nada disso. **O
gradiente descendente depende inteiramente da escala dos dados.** É por isso que padronizar
vem antes de treinar — e a próxima seção mostra isso num conjunto de verdade.

## 5. Muitos atributos: o gradiente em uma linha

Com $d$ atributos, escrever uma derivada por parâmetro não escala. Mas repare que todas têm
a mesma forma, $-\frac{2}{n}\sum_i e_i\,x_{ij}$, e empilhá-las é multiplicar por $X^\top$:

$$\nabla_w J = -\frac{2}{n}\,X^\top (y - Xw)$$

A função abaixo funciona para **qualquer** número de colunas. Primeiro, confira que ela
reproduz os três pontos.

In [ ]:
def gd(X, y, eta, iteracoes):
    n, d = X.shape
    w = np.zeros(d)
    perdas = []
    for _ in range(iteracoes):
        e = y - X @ w
        perdas.append(np.mean(e**2))
        w = w - eta * (-2/n) * X.T @ e      # w ← w − η ∇J
    return w, np.array(perdas)

w, _ = gd(X, y, eta=0.1, iteracoes=2000)
print("versão matricial:", w)
print("forma fechada:   ", w_exato)

### 5.1 Casas da Califórnia

20.640 quarteirões do censo de 1990. Vamos prever o **valor mediano das casas** (em centenas
de milhares de dólares) a partir de quatro atributos.

Olhe as escalas antes de qualquer coisa: renda na casa das unidades, número de cômodos na
casa dos milhares.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula03/housing.csv"
df = pd.read_csv(url)

atributos = ['median_income', 'housing_median_age', 'total_rooms', 'population']
A = df[atributos].to_numpy(dtype=float)
alvo = df['median_house_value'].to_numpy(dtype=float) / 1e5

print(df[atributos].describe().loc[['mean', 'std', 'min', 'max']].round(1))

# gabarito: mínimos quadrados direto, para comparar
X_bruto = np.column_stack([np.ones(len(A)), A])
w_ref = np.linalg.lstsq(X_bruto, alvo, rcond=None)[0]
rmse_ref = np.sqrt(np.mean((alvo - X_bruto @ w_ref)**2))
print(f"\nRMSE da melhor reta possível (gabarito): {rmse_ref:.4f}")

### 5.2 Sem padronizar

> ✋ **Aposte antes de rodar.** $\eta = 0{,}01$ funcionou bem com os três pontos. E aqui?

In [ ]:
with np.errstate(over='ignore', invalid='ignore'):
    w, perdas = gd(X_bruto, alvo, eta=0.01, iteracoes=10)

for it, p in enumerate(perdas):
    print(f"iteração {it}:  J = {p:.3g}")

Explodiu em poucas iterações. A coluna `total_rooms` chega a 39 mil; o gradiente na
direção dela é gigantesco e qualquer passo razoável para `median_income` é um salto absurdo
para `total_rooms`.

A saída ingênua é diminuir $\eta$ até parar de explodir. Veja o que isso custa.

In [ ]:
w, perdas = gd(X_bruto, alvo, eta=1e-8, iteracoes=1000)

print(f"RMSE após 1000 iterações: {np.sqrt(perdas[-1]):.4f}   (gabarito: {rmse_ref:.4f})")
print(f"\npeso da renda — gd: {w[1]:.5f}   gabarito: {w_ref[1]:.5f}")

Não explode, mas também não aprende: o $\eta$ pequeno o bastante para `total_rooms` é
pequeno demais para todo o resto. O peso da renda, o atributo que mais importa, mal saiu
do zero. É o vale estreito da seção 4, agora em cinco dimensões.

### 5.3 Padronizando

Deixe cada coluna com média 0 e desvio 1 — a versão com vários atributos de centrar o $x$.

*(Aqui usamos o conjunto inteiro porque o assunto é otimização. Num modelo de verdade, média
e desvio saem **só do treino**, dentro de um pipeline.)*

In [ ]:
Z = (A - A.mean(axis=0)) / A.std(axis=0)
X_pad = np.column_stack([np.ones(len(Z)), Z])

w, perdas = gd(X_pad, alvo, eta=0.1, iteracoes=200)
w_ref_pad = np.linalg.lstsq(X_pad, alvo, rcond=None)[0]

print(f"RMSE após 200 iterações: {np.sqrt(perdas[-1]):.4f}   (gabarito: {rmse_ref:.4f})")
print("\npesos — gd:      ", w)
print("pesos — gabarito:", w_ref_pad)

plt.figure(figsize=(6, 3.2))
plt.plot(np.sqrt(perdas), color=AZUL, lw=2)
plt.axhline(rmse_ref, color=VERMELHO, ls='--', lw=1, label='gabarito')
plt.xlabel('iteração'); plt.ylabel('RMSE'); plt.legend(); plt.grid(alpha=.3)
plt.title('padronizado, η = 0,1'); plt.show()

Mesmos dados, mesmo algoritmo, mesmo código. A única mudança foi a escala das colunas, e o
método saiu de "explode ou não anda" para "chega ao gabarito em umas 20 iterações".

Com as colunas padronizadas, os pesos também ficam comparáveis: a renda domina, e
`population` entra com sinal negativo.

## 6. Mini-batch: passos baratos

Cada iteração da seção 5 percorre as 20.640 linhas para dar **um** passo. Com milhões de
linhas isso fica caro.

A alternativa é estimar o gradiente com um **lote pequeno** de linhas sorteadas e dar
muitos passos baratos. Cada estimativa é ruidosa, mas aponta, na média, para o lugar certo.

Uma **época** é uma passada completa pelos dados. Com lotes de 32, uma época tem ~645 passos.

In [ ]:
def minibatch_gd(X, y, eta, epocas, tamanho_lote, semente=0):
    rng = np.random.default_rng(semente)
    n, d = X.shape
    w = np.zeros(d)
    perdas = []
    for _ in range(epocas):
        ordem = rng.permutation(n)                     # embaralha a cada época
        for inicio in range(0, n, tamanho_lote):
            lote = ordem[inicio:inicio + tamanho_lote]
            e = y[lote] - X[lote] @ w
            w = w - eta * (-2/len(lote)) * X[lote].T @ e
            perdas.append(np.mean((y - X @ w)**2))     # J no conjunto todo, só para plotar
    return w, np.array(perdas)

w_mb, perdas_mb = minibatch_gd(X_pad, alvo, eta=0.01, epocas=1, tamanho_lote=32)
w_batch, _ = gd(X_pad, alvo, eta=0.1, iteracoes=1)
rmse_batch = np.sqrt(np.mean((alvo - X_pad @ w_batch)**2))

print("depois de UMA passada pelos dados:")
print(f"  {'batch GD, 1 passo':<26} RMSE = {rmse_batch:.4f}")
print(f"  {f'mini-batch, {len(perdas_mb)} passos':<26} RMSE = {np.sqrt(perdas_mb[-1]):.4f}")
print(f"  {'gabarito':<26} RMSE = {rmse_ref:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
passos = np.arange(len(perdas_mb))
for ax, inicio, titulo in [(ax1, 0, 'uma época de mini-batch'), (ax2, 150, 'zoom a partir do passo 150')]:
    ax.plot(passos[inicio:], np.sqrt(perdas_mb[inicio:]), color=LARANJA, lw=1, label='mini-batch (lote 32)')
    ax.axhline(rmse_ref, color=VERMELHO, ls='--', lw=1, label='gabarito')
    ax.set_xlabel('passo'); ax.set_ylabel('RMSE (conjunto todo)')
    ax.set_title(titulo); ax.grid(alpha=.3)
ax1.legend()
plt.tight_layout(); plt.show()

Com o mesmo custo de uma iteração do método em lote — uma passada pelos dados — o
mini-batch já chegou perto do gabarito. No zoom da direita aparece o preço: a curva
**treme** em volta do piso em vez de assentar nele, porque cada passo vê só 32 casas e
cada lote puxa um pouco para um lado. Barato e ruidoso, mas no rumo certo.

É esse laço, com perdas diferentes no lugar do erro quadrático, que treina regressão
logística, SVMs e redes neurais. O algoritmo é o mesmo que você escreveu na seção 1.

## 7. Exercícios

1. **O caso da fronteira.** Na seção 2, rode `descer(0.0, 1.0, 12)`. Antes, use o fator
   $(1 - 2\eta)$ para prever o que vai sair. O que acontece com a distância até o mínimo?

2. **Outro ponto de partida.** Na seção 3.3, chame `treinar(x, y, 0.1, 2000, w0=10, w1=-10)`.
   A resposta final muda? Por que, para esta perda, o ponto de partida não importa?
   (Dica: quantos mínimos a paisagem da seção 4 tem?)

3. **Onde fica o limite.** Com os dados originais da seção 4, encontre o maior $\eta$ que
   ainda converge, com duas casas decimais. Comece testando 0,17, 0,18 e 0,19. Repita com
   `x_c`. Quantas vezes maior é o limite depois de centrar?

4. **Todos os atributos.** Na seção 5, troque a lista `atributos` por todas as colunas
   numéricas, exceto o alvo. Uma delas tem valores ausentes: remova essas linhas com
   `df = df.dropna()` antes de montar `A`. O RMSE do gabarito melhora? O gradiente
   descendente padronizado ainda chega lá?

5. **Tamanho do lote.** Na seção 6, rode `minibatch_gd` com `tamanho_lote` igual a 1, 32 e
   1024, uma época cada, e plote as três curvas. Qual treme mais? Qual chega mais perto do
   gabarito com uma passada? Para `tamanho_lote=1` talvez seja preciso reduzir $\eta$ — por quê?